# AgriWatch Monitor — anomalie suszy dla winnicy (Condom, Gers)
Otwieraj z GitHub (Plik → Otwórz → GitHub). Środowisko z **GPU** (dla SEN2SR). Potem `Uruchom wszystko`.

Łańcuch: Sentinel-2 (rastry, jak w v1) → SEN2SR 2,5 m z kontrolą → indeksy winnicy → anomalie (ERA5-Land SMA, SPI, NDVI)
→ status dekadowy wg logiki EDO CDI → walidacja na ISMN Condom → biuletyn i JSON dla strony.
Wyniki: `MyDrive/GeoWorldLook/agriwatch/data/05_Final_Outputs/agriwatch/` i tabele `data/registry/gwl_*.csv`.


## 1. Dysk Google i kod z GitHub


In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = 'https://github.com/geoworldlook/1_geoworldlook-agriscreen.git'
PROJECT_DIR = '/content/drive/MyDrive/GeoWorldLook/agriwatch'
if IN_COLAB:
    if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
        os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
        !git clone {REPO_URL} "{PROJECT_DIR}"
    !git -C "{PROJECT_DIR}" pull --ff-only
else:
    PROJECT_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import importlib
sys.modules.setdefault('imp', importlib)
try:
    %load_ext autoreload
    %autoreload 2
except Exception as e:
    print(f'[INFO] autoreload pominięty ({e})')
print('Projekt:', PROJECT_DIR)


## 2. Środowisko: pakiety, GEE, rejestr, konfiguracja


In [ ]:
from step_05_colab_run import (setup_runtime, monitor_config, run_task, run_history, registry_summary,
                               task_ingest_s2, task_ingest_era5, task_scene_stats, task_anomalies,
                               task_validate, task_bulletin)
rt = setup_runtime(PROJECT_DIR, gee_project='ee-geoworldlook', pull=False)
cfg = monitor_config(rt)   # zmiany: monitor_config(rt, {'VINEYARDS': {'VINEYARD_06': 6, 'VINEYARD_07': 7}})


## 3. Dane: rastry Sentinel-2 (przyrostowo) i ERA5-Land 1991 → dziś


In [ ]:
run_task(rt, 'ingest_s2', task_ingest_s2, rt, cfg)
run_task(rt, 'ingest_era5', task_ingest_era5, rt, cfg)


## 4. Indeksy winnicy i stacji: 10 m oraz SEN2SR 2,5 m (kontrola H-SR0/H-SR1, bez zastępstwa interpolacją)


In [ ]:
run_task(rt, 'scene_stats', task_scene_stats, rt, cfg)


## 5. Anomalie, status, walidacja i biuletyn


In [ ]:
run_task(rt, 'anomalies', task_anomalies, rt, cfg)
run_task(rt, 'validate', task_validate, rt, cfg)
run_task(rt, 'bulletin', task_bulletin, rt, cfg)


## 6. Panel


In [ ]:
from IPython.display import display, Image, Markdown
out = cfg['OUTPUT_DIR']
for f in ('last_12_months.png', f"season_{cfg['SHOWCASE_SEASON']}.png"):
    p = os.path.join(out, f)
    if os.path.exists(p):
        display(Image(filename=p))
p = os.path.join(out, 'bulletin.md')
if os.path.exists(p):
    display(Markdown(open(p, encoding='utf-8').read()))
display(registry_summary(rt))
display(run_history(rt, n=10))
